# Advanced Problems with Solutions: Variables are Memory References

This notebook explores the idea that Python variables hold **references to objects**, not the objects themselves.

## Learning goals
- Predict when two variables reference the same object.
- Distinguish **rebinding** from **mutation**.
- Use `id()` carefully and interpret it correctly.
- Understand aliasing, function arguments, and copying.
- Avoid common mistakes involving `is`, `==`, and mutable defaults.

## Important note
- `id()` returns an identifier for an object during its lifetime.
- In CPython, this is typically the memory address, but conceptually you should think of it as an **object identity**.
- Do **not** write production code that depends on specific numeric `id()` values.

## Refresher Example

In [1]:
my_var = 10
greeting = 'Hello'

print('my_var =', my_var)
print('id(my_var) =', id(my_var))
print('hex(id(my_var)) =', hex(id(my_var)))
print()
print('greeting =', greeting)
print('id(greeting) =', id(greeting))
print('hex(id(greeting)) =', hex(id(greeting)))

my_var = 10
id(my_var) = 140722596377800
hex(id(my_var)) = 0x7ffc885e74c8

greeting = Hello
id(greeting) = 1871020896592
hex(id(greeting)) = 0x1b3a1893150


The exact numbers will vary, but the key point is:

- `my_var` refers to an integer object.
- `greeting` refers to a string object.
- Different objects usually have different identities.

---

# Problems

## Problem 1 — Rebinding vs. Shared Reference

Predict the output before running the code.

1. Do `a` and `b` initially reference the same object?
2. After rebinding `a`, do they still reference the same object?
3. What happens to `id(a)` and `id(b)`?

In [2]:
a = [1, 2, 3]
b = a

print('Initially:')
print('a =', a, 'id(a) =', id(a))
print('b =', b, 'id(b) =', id(b))
print('a is b ->', a is b)

a = [1, 2, 3, 4]

print('\nAfter rebinding a:')
print('a =', a, 'id(a) =', id(a))
print('b =', b, 'id(b) =', id(b))
print('a is b ->', a is b)

Initially:
a = [1, 2, 3] id(a) = 1871020739328
b = [1, 2, 3] id(b) = 1871020739328
a is b -> True

After rebinding a:
a = [1, 2, 3, 4] id(a) = 1871020847936
b = [1, 2, 3] id(b) = 1871020739328
a is b -> False


### Solution 1

Initially, `b = a` makes `b` reference the **same list object** as `a`, so `a is b` is `True`.

When we later execute:

```python
a = [1, 2, 3, 4]
```

we are **rebinding** `a` to a new list object. We are **not** modifying the original list.

So:
- `id(a)` changes.
- `id(b)` stays the same.
- `b` still points to the original `[1, 2, 3]`.
- `a is b` becomes `False`.

**Core idea:** assignment can change what a variable references without changing the original object.

## Problem 2 — Mutation Through an Alias

This time, predict what changes after calling `append`.

1. Does `append` rebind the variable or mutate the object?
2. What do `x` and `y` print afterward?
3. Do the identities change?

In [3]:
x = ['A', 'B']
y = x

print('Before mutation:')
print('x =', x, 'id(x) =', id(x))
print('y =', y, 'id(y) =', id(y))

x.append('C')

print('\nAfter x.append(\'C\'):')
print('x =', x, 'id(x) =', id(x))
print('y =', y, 'id(y) =', id(y))
print('x is y ->', x is y)

Before mutation:
x = ['A', 'B'] id(x) = 1871021112960
y = ['A', 'B'] id(y) = 1871021112960

After x.append('C'):
x = ['A', 'B', 'C'] id(x) = 1871021112960
y = ['A', 'B', 'C'] id(y) = 1871021112960
x is y -> True


### Solution 2

`y = x` makes `x` and `y` aliases for the same list.

Then:

```python
x.append('C')
```

mutates the existing list object in place. No new list is created.

Therefore:
- Both `x` and `y` print `['A', 'B', 'C']`.
- `id(x)` and `id(y)` remain the same.
- `x is y` is still `True`.

**Core idea:** mutation changes the object that multiple names may share.

## Problem 3 — Immutable Objects and `+=`

Predict the behavior for integers.

Why does `n += 1` behave differently from `list.append(...)`?

In [4]:
n = 10
m = n

print('Before:')
print('n =', n, 'id(n) =', id(n))
print('m =', m, 'id(m) =', id(m))
print('n is m ->', n is m)

n += 1

print('\nAfter n += 1:')
print('n =', n, 'id(n) =', id(n))
print('m =', m, 'id(m) =', id(m))
print('n is m ->', n is m)

Before:
n = 10 id(n) = 140722596377800
m = 10 id(m) = 140722596377800
n is m -> True

After n += 1:
n = 11 id(n) = 140722596377832
m = 10 id(m) = 140722596377800
n is m -> False


### Solution 3

Integers are **immutable**. That means Python cannot change the integer object `10` into `11` in place.

So when we do:

```python
n += 1
```

Python creates a new integer object for `11` and rebinds `n` to it.

As a result:
- `m` still refers to `10`.
- `n` now refers to `11`.
- `id(n)` changes.
- `id(m)` does not change.

**Core idea:** for immutable objects, operations that appear to “change” a value usually create a new object instead.

## Problem 4 — Function Arguments: Rebinding Inside a Function

Predict what prints inside and outside the function.

Does assigning to the parameter `lst` affect the caller's variable?

In [5]:
def rebind_list(lst):
    print('Inside function, before rebinding:', lst, 'id(lst) =', id(lst))
    lst = [99, 100]
    print('Inside function, after rebinding :', lst, 'id(lst) =', id(lst))

data = [1, 2, 3]
print('Outside before call:', data, 'id(data) =', id(data))
rebind_list(data)
print('Outside after call :', data, 'id(data) =', id(data))

Outside before call: [1, 2, 3] id(data) = 1871020964032
Inside function, before rebinding: [1, 2, 3] id(lst) = 1871020964032
Inside function, after rebinding : [99, 100] id(lst) = 1871020958656
Outside after call : [1, 2, 3] id(data) = 1871020964032


### Solution 4

When `rebind_list(data)` is called, the parameter `lst` initially refers to the same object as `data`.

But this line:

```python
lst = [99, 100]
```

only rebinds the **local name** `lst` to a new list.

It does **not** change what `data` refers to in the caller.

So after the function call:
- `data` is still `[1, 2, 3]`.
- The `id(data)` outside the function is unchanged.

**Core idea:** function parameters are local names. Rebinding a parameter does not rebind the caller's variable.

## Problem 5 — Function Arguments: Mutation Inside a Function

Now compare this with in-place mutation.

In [6]:
def mutate_list(lst):
    print('Inside function, before mutation:', lst, 'id(lst) =', id(lst))
    lst.append(999)
    print('Inside function, after mutation :', lst, 'id(lst) =', id(lst))

values = [10, 20]
print('Outside before call:', values, 'id(values) =', id(values))
mutate_list(values)
print('Outside after call :', values, 'id(values) =', id(values))

Outside before call: [10, 20] id(values) = 1871020904064
Inside function, before mutation: [10, 20] id(lst) = 1871020904064
Inside function, after mutation : [10, 20, 999] id(lst) = 1871020904064
Outside after call : [10, 20, 999] id(values) = 1871020904064


### Solution 5

The parameter `lst` and the caller's variable `values` initially refer to the same list object.

This line:

```python
lst.append(999)
```

mutates that shared object in place.

Therefore the change is visible outside the function as well:
- `values` becomes `[10, 20, 999]`.
- The identity stays the same because the same list object was modified.

**Core idea:** rebinding inside a function is local, but mutating a shared mutable object affects all references to that object.

## Problem 6 — `==` vs `is`

Explain the difference between equality and identity.

Predict the results below.

In [7]:
p = [1, 2, 3]
q = [1, 2, 3]
r = p

print('p == q ->', p == q)
print('p is q ->', p is q)
print('p == r ->', p == r)
print('p is r ->', p is r)

p == q -> True
p is q -> False
p == r -> True
p is r -> True


### Solution 6

- `==` checks whether two objects have the same **value/content**.
- `is` checks whether two variables reference the **same object**.

So:
- `p == q` is `True` because both lists contain the same elements.
- `p is q` is `False` because they are distinct list objects.
- `p == r` is `True` because they have the same content.
- `p is r` is `True` because `r = p` makes them aliases.

**Best practice:** use `==` for value comparison. Use `is` mainly for identity checks such as `x is None`.

## Problem 7 — Nested Lists and Shallow Copy

This problem is designed to expose a subtle bug.

Predict what changes after mutating `copied[0]`.

In [8]:
original = [[1, 2], [3, 4]]
copied = original.copy()

print('Before mutation:')
print('original =', original, 'id(original) =', id(original))
print('copied   =', copied,   'id(copied)   =', id(copied))
print('original[0] is copied[0] ->', original[0] is copied[0])

copied[0].append(99)

print('\nAfter copied[0].append(99):')
print('original =', original)
print('copied   =', copied)
print('original[0] is copied[0] ->', original[0] is copied[0])

Before mutation:
original = [[1, 2], [3, 4]] id(original) = 1871020925376
copied   = [[1, 2], [3, 4]] id(copied)   = 1871020925312
original[0] is copied[0] -> True

After copied[0].append(99):
original = [[1, 2, 99], [3, 4]]
copied   = [[1, 2, 99], [3, 4]]
original[0] is copied[0] -> True


### Solution 7

`original.copy()` makes a **shallow copy** of the outer list only.

That means:
- `original` and `copied` are different outer lists.
- But their inner lists are still shared references.

So `original[0] is copied[0]` is `True`.

When we mutate:

```python
copied[0].append(99)
```

the shared inner list changes, so **both** `original` and `copied` appear to change.

**Core idea:** a shallow copy duplicates only one level of structure.

## Problem 8 — Deep Copy

Use `deepcopy` to avoid the issue from Problem 7.

In [9]:
from copy import deepcopy

nested = [[1, 2], [3, 4]]
deep = deepcopy(nested)

print('Before mutation:')
print('nested =', nested)
print('deep   =', deep)
print('nested is deep ->', nested is deep)
print('nested[0] is deep[0] ->', nested[0] is deep[0])

deep[0].append('X')

print('\nAfter deep[0].append(\'X\'):')
print('nested =', nested)
print('deep   =', deep)
print('nested[0] is deep[0] ->', nested[0] is deep[0])

Before mutation:
nested = [[1, 2], [3, 4]]
deep   = [[1, 2], [3, 4]]
nested is deep -> False
nested[0] is deep[0] -> False

After deep[0].append('X'):
nested = [[1, 2], [3, 4]]
deep   = [[1, 2, 'X'], [3, 4]]
nested[0] is deep[0] -> False


### Solution 8

`deepcopy` recursively copies nested mutable objects.

So:
- `nested is deep` is `False`.
- `nested[0] is deep[0]` is also `False`.

After mutating `deep[0]`, the original `nested` structure remains unchanged.

**Best practice:** when nested mutable objects must be fully independent, use `copy.deepcopy(...)`.

## Problem 9 — A Surprising Function Default

This is a classic advanced pitfall.

Predict the outputs of the two calls.

In [10]:
def add_item(item, bucket=[]):
    bucket.append(item)
    return bucket

print(add_item('a'))
print(add_item('b'))

['a']
['a', 'b']


### Solution 9

The default list `bucket=[]` is created **once**, when the function is defined, not each time the function is called.

So both calls reuse the same list object:
- First call returns `['a']`
- Second call returns `['a', 'b']`

This happens because the same mutable object is being referenced across calls.

A safer version is:

In [11]:
def add_item_safe(item, bucket=None):
    if bucket is None:
        bucket = []
    bucket.append(item)
    return bucket

print(add_item_safe('a'))
print(add_item_safe('b'))

['a']
['b']


**Best practice:** never use a mutable object as a default argument unless you intentionally want shared state.

## Problem 10 — Tuples That Contain Mutable Objects

A tuple is immutable, but can it still *indirectly* expose mutation?

In [12]:
t = ([1, 2], 'hello')
print('Before:', t)

t[0].append(3)
print('After :', t)

try:
    t[1] = 'world'
except TypeError as e:
    print('Error:', e)

Before: ([1, 2], 'hello')
After : ([1, 2, 3], 'hello')
Error: 'tuple' object does not support item assignment


### Solution 10

The tuple object itself is immutable, which means its slots cannot be reassigned.

However, one of its elements is a list, and that list is mutable.

So:
- `t[0].append(3)` works because it mutates the list object inside the tuple.
- `t[1] = 'world'` fails because it tries to reassign an element of the tuple itself.

**Core idea:** immutability of a container does not automatically make all contained objects immutable.

## Problem 11 — Object Identity Through a Function Return

Will the returned object be the same one that was passed in?

In [13]:
def process(items):
    items.append('done')
    return items

job = ['start']
result = process(job)

print('job    =', job,    'id(job)    =', id(job))
print('result =', result, 'id(result) =', id(result))
print('job is result ->', job is result)

job    = ['start', 'done'] id(job)    = 1871020867840
result = ['start', 'done'] id(result) = 1871020867840
job is result -> True


### Solution 11

Yes. The function mutates the list it receives and then returns that same object.

Therefore:
- `job` and `result` reference the same list.
- `job is result` is `True`.
- The contents of `job` become `['start', 'done']`.

**Core idea:** returning an object does not imply creating a new one.

## Problem 12 — Build a Mental Model

For the code below, write a step-by-step explanation of what each variable references after each line.

In [14]:
a = [1, 2]
b = a
c = a.copy()
a.append(3)
b = ['new']

print('a =', a, 'id(a) =', id(a))
print('b =', b, 'id(b) =', id(b))
print('c =', c, 'id(c) =', id(c))

a = [1, 2, 3] id(a) = 1871020924160
b = ['new'] id(b) = 1871020750336
c = [1, 2] id(c) = 1871020927232


### Solution 12

Step by step:

1. `a = [1, 2]`
   - `a` references a new list object `L1`.

2. `b = a`
   - `b` now also references `L1`.

3. `c = a.copy()`
   - `c` references a new list object `L2` with the same contents as `L1`.
   - `c` is not the same object as `a`.

4. `a.append(3)`
   - This mutates `L1` in place.
   - Since `b` also refers to `L1`, both `a` and `b` would reflect the change at this moment.

5. `b = ['new']`
   - `b` is rebound to a new list object `L3`.
   - `a` still references `L1`.
   - `c` still references `L2`.

Final values:
- `a` is `[1, 2, 3]`
- `b` is `['new']`
- `c` is `[1, 2]`

**Core idea:** one program can contain rebinding, aliasing, copying, and mutation all at once.

# Summary of Best Practices

1. Think of variables as **labels pointing to objects**.
2. Distinguish clearly between:
   - **rebinding a variable**
   - **mutating an object**
3. Use `==` for value comparison.
4. Use `is` mainly for identity checks such as `x is None`.
5. Be careful when multiple variables reference the same mutable object.
6. Use shallow copies and deep copies intentionally.
7. Avoid mutable default arguments.
8. Use `id()` for learning and debugging, not as normal program logic.

# Optional Challenge Problems

Try solving these on your own before writing code:

1. Why does `s += '!'` for strings usually change the identity of `s`?
2. How would you prove that two nested dictionaries still share inner objects after a shallow copy?
3. Can two immutable objects with the same value ever have different identities?
4. Why is it dangerous to rely on `id()` values across different program runs?